# PetShop Huellitas — Auditoría y preparación de datos

Este notebook implementa un flujo reproducible de auditoría, limpieza inicial, validación e integridad referencial.

Principios del proceso:

- Los archivos de `datos/raw` nunca se modifican.
- Todas las transformaciones se realizan sobre copias.
- Los registros problemáticos se separan para revisión; no desaparecen silenciosamente.
- Las validaciones finales comprueban que el proceso pueda ejecutarse desde un kernel limpio.

Dependencias del entorno: `pandas`, `numpy`, `openpyxl`, `jupyter` e `ipykernel`.


## 1. Configuración y carga


In [ ]:
from pathlib import Path
import re

import numpy as np
import pandas as pd

pd.set_option("display.max_columns", 50)
pd.set_option("display.max_rows", 100)

DIRECTORIO_ACTUAL = Path.cwd().resolve()
RUTA_PROYECTO = (
    DIRECTORIO_ACTUAL.parent
    if DIRECTORIO_ACTUAL.name.lower() == "notebooks"
    else DIRECTORIO_ACTUAL
)
RUTA_DATOS = RUTA_PROYECTO / "datos" / "raw"
RUTA_PROCESADOS = RUTA_PROYECTO / "datos" / "procesados"

ARCHIVOS_REQUERIDOS = [
    "ventas.csv",
    "productos.xlsx",
    "clientes.xlsx",
    "stock.csv",
]

archivos_faltantes = [
    nombre for nombre in ARCHIVOS_REQUERIDOS
    if not (RUTA_DATOS / nombre).is_file()
]

print("Directorio del proyecto:", RUTA_PROYECTO)
print("Directorio de datos:", RUTA_DATOS)

if archivos_faltantes:
    raise FileNotFoundError(
        "Faltan archivos en datos/raw: " + ", ".join(archivos_faltantes)
    )

print("✓ Los cuatro archivos requeridos están disponibles.")


In [ ]:
ventas = pd.read_csv(
    RUTA_DATOS / "ventas.csv",
    encoding="utf-8-sig",
    dtype={
        "id_venta": "string",
        "id_cliente": "string",
        "id_producto": "string",
    },
)

productos = pd.read_excel(
    RUTA_DATOS / "productos.xlsx",
    dtype={"id_producto": "string"},
)

clientes = pd.read_excel(
    RUTA_DATOS / "clientes.xlsx",
    dtype={"id_cliente": "string"},
)

stock = pd.read_csv(
    RUTA_DATOS / "stock.csv",
    encoding="utf-8-sig",
    dtype={"id_producto": "string"},
)

datasets_originales = {
    "ventas": ventas,
    "productos": productos,
    "clientes": clientes,
    "stock": stock,
}

print("✓ Archivos cargados correctamente.")


## 2. Auditoría inicial


In [ ]:
resumen_calidad = pd.DataFrame([
    {
        "archivo": nombre,
        "filas": len(df),
        "columnas": len(df.columns),
        "filas_duplicadas": int(df.duplicated().sum()),
        "celdas_vacias": int(df.isna().sum().sum()),
        "memoria_mb": round(df.memory_usage(deep=True).sum() / 1024**2, 3),
    }
    for nombre, df in datasets_originales.items()
])

display(resumen_calidad)

for nombre, df in datasets_originales.items():
    print(f"\n{nombre.upper()}: {df.shape[0]} filas × {df.shape[1]} columnas")
    display(df.head())


In [ ]:
resumen_faltantes = []

for nombre, df in datasets_originales.items():
    for columna in df.columns:
        cantidad = int(df[columna].isna().sum())
        if cantidad > 0:
            resumen_faltantes.append({
                "archivo": nombre,
                "columna": columna,
                "cantidad": cantidad,
                "porcentaje": round(cantidad / len(df) * 100, 2),
            })

resumen_faltantes = pd.DataFrame(resumen_faltantes)
display(resumen_faltantes)


## 3. Copias de trabajo y duplicados

Una repetición de `id_venta` no implica un duplicado: una misma venta puede contener varios productos. En esta etapa solo se eliminan filas idénticas en todas sus columnas.


In [ ]:
ventas_limpias = ventas.copy()
productos_limpios = productos.copy()
clientes_limpios = clientes.copy()
stock_limpio = stock.copy()

registro_limpieza = []


def eliminar_duplicados_exactos(df, nombre_tabla):
    filas_antes = len(df)
    resultado = df.drop_duplicates().reset_index(drop=True)
    filas_despues = len(resultado)

    registro_limpieza.append({
        "tabla": nombre_tabla,
        "proceso": "Eliminación de duplicados exactos",
        "filas_antes": filas_antes,
        "filas_despues": filas_despues,
        "filas_afectadas": filas_antes - filas_despues,
        "criterio": "Coincidencia en todas las columnas",
    })
    return resultado


ventas_limpias = eliminar_duplicados_exactos(ventas_limpias, "ventas")
productos_limpios = eliminar_duplicados_exactos(productos_limpios, "productos")
clientes_limpios = eliminar_duplicados_exactos(clientes_limpios, "clientes")
stock_limpio = eliminar_duplicados_exactos(stock_limpio, "stock")

registro_limpieza_df = pd.DataFrame(registro_limpieza)
display(registro_limpieza_df)

for nombre, df in {
    "ventas": ventas_limpias,
    "productos": productos_limpios,
    "clientes": clientes_limpios,
    "stock": stock_limpio,
}.items():
    assert df.duplicated().sum() == 0, f"Quedan duplicados exactos en {nombre}"

print("✓ No quedan duplicados exactos.")


In [ ]:
duplicados_clave_productos = productos_limpios[
    productos_limpios.duplicated("id_producto", keep=False)
]
duplicados_clave_clientes = clientes_limpios[
    clientes_limpios.duplicated("id_cliente", keep=False)
]

assert duplicados_clave_productos.empty, "Hay id_producto repetidos"
assert duplicados_clave_clientes.empty, "Hay id_cliente repetidos"

if "id_linea_venta" not in ventas_limpias.columns:
    ventas_limpias.insert(
        0,
        "id_linea_venta",
        [f"LV{numero:06d}" for numero in range(1, len(ventas_limpias) + 1)],
    )

assert ventas_limpias["id_linea_venta"].is_unique
print("✓ Claves maestras válidas e identificador de línea creado.")


## 4. Normalización de textos


In [ ]:
bitacora_transformaciones = []


def normalizar_espacios(serie):
    return (
        serie.astype("string")
        .str.strip()
        .str.replace(r"\s+", " ", regex=True)
    )


def registrar_transformacion(tabla, columna, proceso, antes, despues):
    cambios = int(
        (antes.fillna("<VACIO>") != despues.fillna("<VACIO>")).sum()
    )
    bitacora_transformaciones.append({
        "tabla": tabla,
        "columna": columna,
        "proceso": proceso,
        "registros_modificados": cambios,
    })


columnas_identificadores = {
    "ventas": (ventas_limpias, ["id_linea_venta", "id_venta", "id_cliente", "id_producto"]),
    "productos": (productos_limpios, ["id_producto"]),
    "clientes": (clientes_limpios, ["id_cliente"]),
    "stock": (stock_limpio, ["id_producto"]),
}

for tabla, (df, columnas) in columnas_identificadores.items():
    for columna in columnas:
        antes = df[columna].copy()
        df[columna] = normalizar_espacios(df[columna]).str.upper()
        registrar_transformacion(
            tabla, columna,
            "Eliminación de espacios y conversión a mayúsculas",
            antes, df[columna],
        )


In [ ]:
mapa_sucursales = {
    "centro": "Centro",
    "sucursal centro": "Centro",
    "city bell": "City Bell",
    "sucursal city bell": "City Bell",
    "los hornos": "Los Hornos",
    "sucursal los hornos": "Los Hornos",
    "ecommerce": "Ecommerce",
}

mapa_categorias = {
    "alimento": "Alimentos",
    "alimentos": "Alimentos",
    "snack": "Snacks",
    "snacks": "Snacks",
    "higiene": "Higiene",
    "accesorio": "Accesorios",
    "accesorios": "Accesorios",
    "salud": "Salud",
}

mapa_ciudades = {
    "la plata": "La Plata",
    "lp": "La Plata",
    "l.p.": "La Plata",
    "city bell": "City Bell",
    "los hornos": "Los Hornos",
    "gonnet": "Gonnet",
    "villa elisa": "Villa Elisa",
    "berisso": "Berisso",
    "ensenada": "Ensenada",
    "tolosa": "Tolosa",
    "ringuelet": "Ringuelet",
}


def aplicar_mapa(df, tabla, columna, mapa, proceso):
    antes = df[columna].copy()
    clave = normalizar_espacios(antes).str.casefold()
    df[columna] = clave.map(mapa).fillna(normalizar_espacios(antes))
    registrar_transformacion(tabla, columna, proceso, antes, df[columna])


aplicar_mapa(ventas_limpias, "ventas", "sucursal", mapa_sucursales, "Unificación de sucursales")
aplicar_mapa(stock_limpio, "stock", "sucursal", mapa_sucursales, "Unificación de sucursales")
aplicar_mapa(productos_limpios, "productos", "categoria", mapa_categorias, "Unificación de categorías")
aplicar_mapa(clientes_limpios, "clientes", "ciudad", mapa_ciudades, "Unificación de ciudades")

for tabla, df, columnas in [
    ("productos", productos_limpios, ["producto", "subcategoria", "marca", "proveedor"]),
    ("clientes", clientes_limpios, ["nombre", "mascota"]),
    ("ventas", ventas_limpias, ["canal", "medio_pago"]),
]:
    for columna in columnas:
        antes = df[columna].copy()
        df[columna] = normalizar_espacios(antes)
        registrar_transformacion(tabla, columna, "Normalización de espacios", antes, df[columna])

email_antes = clientes_limpios["email"].copy()
clientes_limpios["email"] = normalizar_espacios(email_antes).str.lower()
registrar_transformacion(
    "clientes", "email", "Espacios y conversión a minúsculas",
    email_antes, clientes_limpios["email"],
)

bitacora_transformaciones_df = pd.DataFrame(bitacora_transformaciones)
display(bitacora_transformaciones_df.query("registros_modificados > 0"))


In [ ]:
categorias_validas = {"Alimentos", "Snacks", "Higiene", "Accesorios", "Salud"}
sucursales_ventas_validas = {"Centro", "City Bell", "Los Hornos", "Ecommerce"}
sucursales_stock_validas = {"Centro", "City Bell", "Los Hornos"}

assert set(productos_limpios["categoria"].dropna()).issubset(categorias_validas)
assert set(ventas_limpias["sucursal"].dropna()).issubset(sucursales_ventas_validas)
assert set(stock_limpio["sucursal"].dropna()).issubset(sucursales_stock_validas)

print("✓ Textos y categorías normalizados correctamente.")


## 5. Conversión de fechas e importes


In [ ]:
def convertir_fecha(serie):
    return pd.to_datetime(
        serie,
        format="mixed",
        dayfirst=True,
        errors="coerce",
    )


ventas_limpias["fecha_original"] = ventas_limpias["fecha"].copy()
clientes_limpios["fecha_alta_original"] = clientes_limpios["fecha_alta"].copy()
stock_limpio["fecha_original"] = stock_limpio["fecha"].copy()

ventas_limpias["fecha"] = convertir_fecha(ventas_limpias["fecha_original"])
clientes_limpios["fecha_alta"] = convertir_fecha(clientes_limpios["fecha_alta_original"])
stock_limpio["fecha"] = convertir_fecha(stock_limpio["fecha_original"])

inicio_ventas = pd.Timestamp("2025-01-01")
fin_datos = pd.Timestamp("2026-08-31")

ventas_limpias["fecha_invalida"] = ventas_limpias["fecha"].isna()
ventas_limpias["fecha_fuera_rango"] = (
    ventas_limpias["fecha"].notna()
    & ~ventas_limpias["fecha"].between(inicio_ventas, fin_datos)
)

clientes_limpios["fecha_alta_invalida"] = (
    clientes_limpios["fecha_alta"].isna()
    & clientes_limpios["fecha_alta_original"].notna()
)
clientes_limpios["fecha_alta_fuera_rango"] = (
    clientes_limpios["fecha_alta"].notna()
    & (clientes_limpios["fecha_alta"] > fin_datos)
)

stock_limpio["fecha_invalida"] = stock_limpio["fecha"].isna()
stock_limpio["fecha_fuera_rango"] = (
    stock_limpio["fecha"].notna()
    & (stock_limpio["fecha"] != fin_datos)
)

print("✓ Fechas convertidas y validadas.")


In [ ]:
def convertir_importe_argentino(serie):
    def convertir_valor(valor):
        if pd.isna(valor):
            return np.nan
        if isinstance(valor, (int, float, np.number)):
            return float(valor)

        texto = (
            str(valor).strip()
            .replace("$", "")
            .replace(" ", "")
            .replace(".", "")
            .replace(",", ".")
        )
        return pd.to_numeric(texto, errors="coerce")

    return serie.apply(convertir_valor)


for columna in ["cantidad", "precio_unitario", "descuento"]:
    ventas_limpias[columna] = pd.to_numeric(ventas_limpias[columna], errors="coerce")

for columna in ["stock_actual", "stock_minimo"]:
    stock_limpio[columna] = pd.to_numeric(stock_limpio[columna], errors="coerce")

productos_limpios["costo_original"] = productos_limpios["costo"].copy()
productos_limpios["precio_lista_original"] = productos_limpios["precio_lista"].copy()
productos_limpios["costo"] = convertir_importe_argentino(productos_limpios["costo_original"])
productos_limpios["precio_lista"] = convertir_importe_argentino(productos_limpios["precio_lista_original"])

display(productos_limpios[["costo", "precio_lista"]].describe())


## 6. Reglas de calidad y separación de incidencias


In [ ]:
productos_limpios["costo_faltante"] = productos_limpios["costo"].isna()
productos_limpios["precio_invalido"] = (
    productos_limpios["precio_lista"].isna()
    | (productos_limpios["precio_lista"] <= 0)
)
productos_limpios["costo_invalido"] = (
    productos_limpios["costo"].notna()
    & (productos_limpios["costo"] <= 0)
)
productos_limpios["costo_mayor_precio"] = (
    productos_limpios["costo"].notna()
    & productos_limpios["precio_lista"].notna()
    & (productos_limpios["costo"] > productos_limpios["precio_lista"])
)
productos_limpios["requiere_revision"] = productos_limpios[[
    "costo_faltante", "precio_invalido", "costo_invalido", "costo_mayor_precio"
]].any(axis=1)

email_informado = clientes_limpios["email"].notna()
email_valido = clientes_limpios["email"].str.fullmatch(
    r"[^@\s]+@[^@\s]+\.[^@\s]+",
    na=False,
)
clientes_limpios["email_invalido"] = email_informado & ~email_valido
clientes_limpios["ciudad_faltante"] = clientes_limpios["ciudad"].isna()
clientes_limpios["requiere_revision"] = clientes_limpios[[
    "fecha_alta_invalida", "fecha_alta_fuera_rango", "email_invalido", "ciudad_faltante"
]].any(axis=1)

productos_observados = productos_limpios[productos_limpios["requiere_revision"]].copy()
clientes_observados = clientes_limpios[clientes_limpios["requiere_revision"]].copy()

print("Productos para revisar:", len(productos_observados))
print("Clientes para revisar:", len(clientes_observados))


In [ ]:
ventas_limpias["error_cantidad"] = (
    ventas_limpias["cantidad"].isna()
    | (ventas_limpias["cantidad"] <= 0)
)
ventas_limpias["cantidad_atipica"] = ventas_limpias["cantidad"] > 50
ventas_limpias["error_precio"] = (
    ventas_limpias["precio_unitario"].isna()
    | (ventas_limpias["precio_unitario"] <= 0)
)
ventas_limpias["error_descuento"] = (
    ventas_limpias["descuento"].isna()
    | ~ventas_limpias["descuento"].between(0, 1)
)

columnas_error_ventas = [
    "fecha_invalida",
    "fecha_fuera_rango",
    "error_cantidad",
    "cantidad_atipica",
    "error_precio",
    "error_descuento",
]
ventas_limpias["requiere_revision"] = ventas_limpias[columnas_error_ventas].any(axis=1)


def motivos_venta(fila):
    reglas = [
        ("fecha_invalida", "Fecha inválida"),
        ("fecha_fuera_rango", "Fecha fuera de rango"),
        ("error_cantidad", "Cantidad inválida"),
        ("cantidad_atipica", "Cantidad atípica"),
        ("error_precio", "Precio inválido"),
        ("error_descuento", "Descuento inválido"),
    ]
    return ", ".join(texto for columna, texto in reglas if fila[columna])


ventas_limpias["motivo_revision"] = ventas_limpias.apply(motivos_venta, axis=1)
ventas_prevalidas = ventas_limpias[~ventas_limpias["requiere_revision"]].copy().reset_index(drop=True)
ventas_revision = ventas_limpias[ventas_limpias["requiere_revision"]].copy().reset_index(drop=True)

assert len(ventas_prevalidas) + len(ventas_revision) == len(ventas_limpias)
display(ventas_limpias[columnas_error_ventas].sum().rename("cantidad").to_frame())


In [ ]:
stock_limpio["stock_actual_invalido"] = (
    stock_limpio["stock_actual"].isna()
    | (stock_limpio["stock_actual"] < 0)
)
stock_limpio["stock_minimo_invalido"] = (
    stock_limpio["stock_minimo"].isna()
    | (stock_limpio["stock_minimo"] < 0)
)

columnas_error_stock = [
    "fecha_invalida",
    "fecha_fuera_rango",
    "stock_actual_invalido",
    "stock_minimo_invalido",
]
stock_limpio["requiere_revision"] = stock_limpio[columnas_error_stock].any(axis=1)


def motivos_stock(fila):
    reglas = [
        ("fecha_invalida", "Fecha inválida"),
        ("fecha_fuera_rango", "Fecha fuera de rango"),
        ("stock_actual_invalido", "Stock actual inválido"),
        ("stock_minimo_invalido", "Stock mínimo inválido"),
    ]
    return ", ".join(texto for columna, texto in reglas if fila[columna])


stock_limpio["motivo_revision"] = stock_limpio.apply(motivos_stock, axis=1)
stock_prevalido = stock_limpio[~stock_limpio["requiere_revision"]].copy().reset_index(drop=True)
stock_revision = stock_limpio[stock_limpio["requiere_revision"]].copy().reset_index(drop=True)

assert len(stock_prevalido) + len(stock_revision) == len(stock_limpio)
display(stock_limpio[columnas_error_stock].sum().rename("cantidad").to_frame())


## 7. Integridad referencial

- Las ventas sin producto válido se separan porque no pueden enriquecerse con categoría o costo.
- Las ventas sin cliente se conservan como `C_INVITADO`.
- Las ventas con un cliente informado que no existe se conservan como `C_NO_ENCONTRADO`.


In [ ]:
assert productos_limpios["id_producto"].is_unique
assert clientes_limpios["id_cliente"].is_unique

ids_productos = set(productos_limpios["id_producto"].dropna())
ids_clientes = set(clientes_limpios["id_cliente"].dropna())

ventas_prevalidas["producto_existe"] = ventas_prevalidas["id_producto"].isin(ids_productos)
ventas_revision_integridad = ventas_prevalidas[~ventas_prevalidas["producto_existe"]].copy().reset_index(drop=True)
ventas_integras = ventas_prevalidas[ventas_prevalidas["producto_existe"]].copy().reset_index(drop=True)
ventas_revision_integridad["motivo_revision"] = "Producto inexistente en el maestro"

cliente_texto = ventas_integras["id_cliente"].fillna("").astype("string").str.strip()
ventas_integras["cliente_informado"] = cliente_texto.ne("")
ventas_integras["cliente_existe"] = ventas_integras["id_cliente"].isin(ids_clientes)
ventas_integras["estado_cliente"] = np.select(
    [
        ~ventas_integras["cliente_informado"],
        ventas_integras["cliente_existe"],
    ],
    ["Invitado", "Registrado"],
    default="No encontrado",
)
ventas_integras["id_cliente_modelo"] = ventas_integras["id_cliente"].copy()
ventas_integras.loc[ventas_integras["estado_cliente"] == "Invitado", "id_cliente_modelo"] = "C_INVITADO"
ventas_integras.loc[ventas_integras["estado_cliente"] == "No encontrado", "id_cliente_modelo"] = "C_NO_ENCONTRADO"

assert len(ventas_integras) + len(ventas_revision_integridad) == len(ventas_prevalidas)
display(ventas_integras["estado_cliente"].value_counts().rename_axis("estado_cliente").reset_index(name="cantidad_lineas"))


In [ ]:
stock_prevalido["producto_existe"] = stock_prevalido["id_producto"].isin(ids_productos)
stock_revision_integridad = stock_prevalido[~stock_prevalido["producto_existe"]].copy().reset_index(drop=True)
stock_validado = stock_prevalido[stock_prevalido["producto_existe"]].copy().reset_index(drop=True)
stock_revision_integridad["motivo_revision"] = "Producto inexistente en el maestro"

assert len(stock_validado) + len(stock_revision_integridad) == len(stock_prevalido)
print("Stock con producto válido:", len(stock_validado))
print("Stock sin producto válido:", len(stock_revision_integridad))


## 8. Dimensiones y tablas analíticas


In [ ]:
productos_dimension = productos_limpios[[
    "id_producto", "producto", "categoria", "subcategoria",
    "marca", "costo", "precio_lista", "proveedor",
]].copy()

clientes_dimension = clientes_limpios[[
    "id_cliente", "nombre", "ciudad", "fecha_alta", "mascota", "email",
]].copy()

clientes_especiales = pd.DataFrame({
    "id_cliente": ["C_INVITADO", "C_NO_ENCONTRADO"],
    "nombre": ["Cliente invitado", "Cliente no encontrado"],
})
clientes_dimension = pd.concat(
    [clientes_dimension, clientes_especiales],
    ignore_index=True,
)

assert productos_dimension["id_producto"].is_unique
assert clientes_dimension["id_cliente"].is_unique
print("✓ Dimensiones creadas.")


In [ ]:
ventas_enriquecidas = ventas_integras.merge(
    productos_dimension,
    on="id_producto",
    how="left",
    validate="many_to_one",
    indicator="union_producto",
)
assert ventas_enriquecidas["union_producto"].eq("both").all()
ventas_enriquecidas.drop(columns="union_producto", inplace=True)

clientes_para_merge = clientes_dimension.rename(
    columns={"id_cliente": "id_cliente_modelo"}
)
ventas_enriquecidas = ventas_enriquecidas.merge(
    clientes_para_merge,
    on="id_cliente_modelo",
    how="left",
    validate="many_to_one",
    indicator="union_cliente",
)
assert ventas_enriquecidas["union_cliente"].eq("both").all()
ventas_enriquecidas.drop(columns="union_cliente", inplace=True)

stock_enriquecido = stock_validado.merge(
    productos_dimension,
    on="id_producto",
    how="left",
    validate="many_to_one",
    indicator="union_producto",
)
assert stock_enriquecido["union_producto"].eq("both").all()
stock_enriquecido.drop(columns="union_producto", inplace=True)

print("✓ Ventas y stock enriquecidos.")


## 9. Consolidación de incidencias y controles finales


In [ ]:
ventas_observadas = pd.concat(
    [ventas_revision, ventas_revision_integridad],
    ignore_index=True,
    sort=False,
)
stock_observado = pd.concat(
    [stock_revision, stock_revision_integridad],
    ignore_index=True,
    sort=False,
)

resumen_resultado = pd.DataFrame([
    {"tabla": "ventas", "registros_limpios": len(ventas_enriquecidas), "registros_observados": len(ventas_observadas)},
    {"tabla": "stock", "registros_limpios": len(stock_enriquecido), "registros_observados": len(stock_observado)},
    {"tabla": "productos", "registros_limpios": len(productos_dimension), "registros_observados": len(productos_observados)},
    {"tabla": "clientes", "registros_limpios": len(clientes_dimension) - 2, "registros_observados": len(clientes_observados)},
])

assert ventas_enriquecidas["id_linea_venta"].is_unique
assert ventas_enriquecidas["id_producto"].isin(productos_dimension["id_producto"]).all()
assert ventas_enriquecidas["id_cliente_modelo"].isin(clientes_dimension["id_cliente"]).all()
assert stock_enriquecido["id_producto"].isin(productos_dimension["id_producto"]).all()
assert len(ventas_enriquecidas) + len(ventas_observadas) == len(ventas_limpias)
assert len(stock_enriquecido) + len(stock_observado) == len(stock_limpio)

display(resumen_resultado)
print("✓ Notebook ejecutado correctamente de principio a fin.")


## Próximo paso

Con la base validada, el siguiente bloque calculará venta bruta, descuento aplicado, facturación neta, costo total y margen. Los productos sin costo permanecerán identificados para evitar márgenes ficticios.
